# 5G Network Intelligence — Cell Attention Numerical Consistency

## Question
Does the GPU attention inference fastpath explain saved forecast differences versus CPU? Test both backends with fixed checkpoints and origins.


## Protocol
No fit, calibration, checkpoint reselection or holdout. Keep original best weights. Compare three cells at two predeclared origins per fold. Absolute tolerance remains 1e-4 DLPRB units. Reproduce original GPU forecasts first, then require strict-math CPU/GPU parity before full replay. Backend choice is prescribed for consistency, never selected by MAE/WAPE.


In [ ]:
RUN_REPLAY = False


In [ ]:
if RUN_REPLAY:
    # Pin every code/input/checkpoint member before importing modules.
    from pathlib import Path, PurePosixPath
    import hashlib, json, os, shutil, sys
    def verified_manifest(filename, expected):
        matches = [p for p in Path('/kaggle/input').rglob(filename)
                   if hashlib.sha256(p.read_bytes()).hexdigest() == expected]
        if len(matches) != 1:
            raise ValueError('Exactly one pinned manifest required: ' + filename)
        path = matches[0]
        manifest = json.loads(path.read_text())
        for name, digest in manifest['files'].items():
            relative = PurePosixPath(name)
            if relative.is_absolute() or '..' in relative.parts or ':' in name or chr(92) in name:
                raise ValueError('Unsafe member')
            original = path.parent/name
            if not original.resolve().is_relative_to(path.parent.resolve()) or hashlib.sha256(original.read_bytes()).hexdigest() != digest:
                raise ValueError('Member identity differs: ' + name)
        return path, manifest
    code_path, code = verified_manifest('precision_source_manifest.json', 'c92d371b2046157814fbef03780f9b9016ddef774b23e74ecef531ecb7ab4363')
    base_path, base = verified_manifest('source_manifest.json', '72c701d1f226864c1a3ae758dac5c43b18e429c3f8f43dc8e14ea3b208d01af8')
    trained_path, trained = verified_manifest('manifest.json', '0fadd23033ff9228bcf7c76c5f992946a3c599f3efedb88ba4def9975b493788')
    PROJECT = Path('/kaggle/working/cell_precision')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('data', 'configs'):
        shutil.copytree(base_path.parent/folder, PROJECT/folder, dirs_exist_ok=True)
    for name in code['files']:
        target = PROJECT/name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(code_path.parent/name, target)
    TRAINED_ROOT = trained_path.parent
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = code['source_commit']
    print('Verified inference code:', code['source_commit'])
else:
    from pathlib import Path
    PROJECT = Path.cwd()
    print('Review mode: attached-input verification skipped.')


## Backend diagnosis and strict forecast replay
The implementation is `scripts/diagnose_cell_precision.py`. Preserve original artifacts and export new forecasts separately. Training results remain historical snapshots.


In [ ]:
# Frozen checkpoints only: no training or checkpoint selection.
if RUN_REPLAY:
    import subprocess, sys
    (PROJECT/'reports').mkdir(exist_ok=True)
    with (PROJECT/'reports/cell_attention_precision_inference.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.diagnose_cell_precision',
            '--trained-root', str(TRAINED_ROOT)], stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        REPLAY_EXIT = process.wait()
else:
    print('Review only: inference skipped.')


## Limitations
An inference backend fix is not evidence of improved model quality. Independently reconstruct metrics and parity after export. Failure preserves diagnostics and does not relax the tolerance.
